In [1]:
%pip install numpy matplotlib astropy astroquery firefly-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 6.1 MB/s  0:00:02m 6.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 2.9 MB/s  0:00:004.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [astroquery] 3/4 [astroquery]
Note: you may need to restart the kernel to use updated packages.


In [14]:
import http.client
import time
import urllib.error

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
from astropy import units as u
from astropy.coordinates import SkyCoord

from astroquery.ipac.irsa import Irsa

from firefly_client import FireflyClient

# The time it takes to read SPHEREx files can exceed
# astropy's default timeout limit. Increase it.
from astropy.utils.data import conf
conf.remote_timeout = 120

In [3]:
ra_deg = 304.693508808
dec_deg = 42.4436872991

coord = SkyCoord(ra_deg, dec_deg, unit='deg')
search_radius = 1 * u.arcsec

In [4]:
print(coord)

<SkyCoord (ICRS): (ra, dec) in deg
    (304.69350881, 42.4436873)>


In [9]:
results = Irsa.query_sia(pos=(coord, search_radius), collection=['spherex_dr1', 'spherex_qr3'])

In [21]:
results.colnames

['s_ra',
 's_dec',
 'facility_name',
 'instrument_name',
 'dataproduct_subtype',
 'calib_level',
 'dataproduct_type',
 'energy_bandpassname',
 'energy_emband',
 'obs_id',
 's_resolution',
 'em_min',
 'em_max',
 'em_res_power',
 'proposal_title',
 'access_url',
 'access_format',
 'access_estsize',
 't_exptime',
 's_region',
 'obs_collection',
 'obs_intent',
 'algorithm_name',
 'facility_keywords',
 'instrument_keywords',
 'environment_photometric',
 'proposal_id',
 'proposal_pi',
 'proposal_project',
 'target_name',
 'target_type',
 'target_standard',
 'target_moving',
 'target_keywords',
 'obs_release_date',
 's_xel1',
 's_xel2',
 's_pixel_scale',
 'position_timedependent',
 't_min',
 't_max',
 't_resolution',
 't_xel',
 'obs_publisher_did',
 's_fov',
 'em_xel',
 'pol_states',
 'pol_xel',
 'cloud_access',
 'o_ucd',
 'upload_row_id']

In [22]:
spectral_image_url = results["access_url"][0]
print("Spectral Image URL:", spectral_image_url)

Spectral Image URL: https://irsa.ipac.caltech.edu/ibe/data/spherex/dr1/level2/2025W18_1B/l2b-v27-2026-196/1/level2_2025W18_1B_0074_2D1_spx_l2b-v27-2026-196.fits


In [23]:
import os
fits_filename = "level2_spectral_image.fits"
hdulist = fits.open(fits_filename)
hdulist.info()
print("Loaded", fits_filename, "successfully.")

Loaded level2_spectral_image.fits successfully.


In [24]:
from scipy.ndimage import maximum_filter

image_hdu = hdulist["IMAGE"]
image_data = image_hdu.data
header = image_hdu.header

# Celestial WCS
wcs_celestial = WCS(header)
wcs_celestial.sip = None

# Spectral WCS
spectral_wcs = WCS(header=header, fobj=hdulist, key="W")
spectral_wcs.sip = None

# Local peak detection for stars
valid_pixels = image_data[np.isfinite(image_data)]
bg_median = float(np.median(valid_pixels))
bg_std = float(np.std(valid_pixels[valid_pixels < np.percentile(valid_pixels, 95)]))

local_max = (maximum_filter(image_data, size=9) == image_data)
threshold = bg_median + 4.0 * bg_std
detected_mask = local_max & (image_data > threshold) & np.isfinite(image_data)

y_peaks, x_peaks = np.where(detected_mask)
fluxes = image_data[y_peaks, x_peaks]

sort_idx = np.argsort(fluxes)[::-1]
y_peaks = y_peaks[sort_idx][:3500]
x_peaks = x_peaks[sort_idx][:3500]
fluxes = fluxes[sort_idx][:3500]
print(f"Detected {len(fluxes)} stars in SPHEREx field.")

Detected 3500 stars in SPHEREx field.


In [25]:
# Convert to 3D Cartesian coordinates for Three.js / Stellarium
R = 1000.0
coords = wcs_celestial.pixel_to_world(x_peaks, y_peaks)
wls, bws = spectral_wcs.pixel_to_world(x_peaks, y_peaks)

stars_data = []
for i in range(len(fluxes)):
    ra_val = float(coords.ra.deg[i])
    dec_val = float(coords.dec.deg[i])
    fl = float(fluxes[i])
    wl = float(wls.value[i])
    bw = float(bws.value[i])
    
    ra_rad = np.radians(ra_val)
    dec_rad = np.radians(dec_val)
    
    # Three.js 3D coordinates (Y is UP)
    x_3d = float(R * np.cos(dec_rad) * np.sin(ra_rad))
    y_3d = float(R * np.sin(dec_rad))
    z_3d = float(R * np.cos(dec_rad) * np.cos(ra_rad))
    
    mag = float(18.0 - 2.5 * np.log10(max(fl, 1e-4)))
    
    if wl < 0.85:
        color = "#AFC8FF"
    elif wl < 0.95:
        color = "#FFF4E8"
    elif wl < 1.05:
        color = "#FFE4B5"
    else:
        color = "#FF9A60"
        
    stars_data.append({
        "id": f"SPX-{i+1:05d}",
        "name": f"SPHEREx Star {i+1}",
        "ra": round(ra_val, 6),
        "dec": round(dec_val, 6),
        "magnitude": round(mag, 2),
        "flux_mjy_sr": round(fl, 3),
        "color": color,
        "color_hex": color,
        "size": round(max(1.5, min(12.0, 14.0 - mag * 1.2)), 2),
        "position": {"x": round(x_3d, 3), "y": round(y_3d, 3), "z": round(z_3d, 3)},
        "spectral": {"wavelength_um": round(wl, 4), "bandwidth_um": round(bw, 4)}
    })

os.makedirs("data", exist_ok=True)
notebook_export = {
    "source": "file.ipynb",
    "target_coord": {"ra": float(ra_deg), "dec": float(dec_deg)},
    "search_radius_arcsec": 1.0,
    "collection": ["spherex_dr1", "spherex_qr3"],
    "total_stars": len(stars_data),
    "stars": stars_data
}

with open("data/notebook_stars.json", "w") as f:
    json.dump(notebook_export, f, indent=2)
print(f"Successfully exported {len(stars_data)} stars from notebook to data/notebook_stars.json")

Successfully exported 3500 stars from notebook to data/notebook_stars.json
